# Benchmarking

In [25]:
import json
from pathlib import Path

#locals
# from code.run_types import ResSet, RunRes
import pandas as pd

from src.resulting import RESULTS_DIR

In [26]:
def load_mvit_benchmarks(results_dir):
    """
    Load benchmark.json, keep only MViTv2 variants, drop OOM runs,
    map model names, and return separate DataFrames for training and inference.
    """
    results_dir = Path(results_dir)
    with open(results_dir / "benchmark.json", "r") as f:
        raw = json.load(f)

    runs = raw["runs"]

    # Model name mapping
    model_name_map = {
        "MViTv2_S": "MViTv2\\_S",
        "MViTv2_B_32x3": "MViTv2\\_B\\_32x3",
        "MViTv2_S_e": "MViTv2\\_S\\_e",
        "MViTv2_S_16x4": "MViTv2\\_S\\_16x4",
        "MViTv2_S_16x4_e": "MViTv2\\_S\\_16x4\\_e"
    }

    records = []

    for run in runs.values():
        arch = run.get("arch", "")
        if arch not in model_name_map:
            continue
        if "error" in run:
            continue

        config = run["config"]
        results = run["results"]

        records.append({
            "model": model_name_map[arch],  # mapped name
            "num_frames": config["num_frames"],
            "mode": "Train" if config.get("full_step", False) else "Infer",
            "batch_size": config["batch_size"],

            "gpu_util_mean": results["gpu_utilisation_percent"]["mean"],
            "gpu_util_std": results["gpu_utilisation_percent"]["std"],

            "latency_ms_mean": results["latency_ms"]["mean"],
            "latency_ms_std": results["latency_ms"]["std"],

            "throughput_samp_per_s_mean": results["throughput_samples_per_s"]["mean"],
            "throughput_samp_per_s_std": results["throughput_samples_per_s"]["std"],

            "peak_mem_mb_mean": results["peak_memory_mb"]["mean"],
            "peak_mem_mb_std": results["peak_memory_mb"]["std"],
        })

    df = pd.DataFrame(records)

    metric_cols = [
        "gpu_util_mean", "gpu_util_std",
        "latency_ms_mean", "latency_ms_std",
        "throughput_samp_per_s_mean", "throughput_samp_per_s_std",
        "peak_mem_mb_mean", "peak_mem_mb_std",
    ]
    df[metric_cols] = df[metric_cols].round(2)

    # Sort by num_frames, then batch_size
    train_df = df[df["mode"] == "Train"].sort_values(
        ["num_frames", "latency_ms_mean","batch_size"]
    ).reset_index(drop=True)
    infer_df = df[df["mode"] == "Infer"].sort_values(
        ["num_frames","latency_ms_mean", "batch_size"]
    ).reset_index(drop=True)

    return train_df, infer_df

## Load and display results

In [27]:
results_dir = RESULTS_DIR / 'sacair_2026'
train_df, infer_df = load_mvit_benchmarks(results_dir)

print("Training DataFrame:")
display(train_df)

print("\nInference DataFrame:")
display(infer_df)

Training DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv2\_S,16,Train,1,98.35,0.43,175.30,0.26,5.70,0.01,1802.01,0.0
1,MViTv2\_S\_16x4,16,Train,1,98.65,0.04,224.36,0.08,4.46,0.00,1800.72,0.0
2,MViTv2\_S,16,Train,2,99.26,0.01,335.94,0.56,5.95,0.01,3294.64,0.0
3,MViTv2\_S\_16x4,16,Train,2,99.47,0.04,430.14,0.08,4.65,0.00,3293.91,0.0
4,MViTv2\_S,16,Train,4,99.68,0.02,642.09,0.85,6.23,0.01,6224.38,0.0
5,MViTv2\_S\_16x4,16,Train,4,99.79,0.02,826.59,0.10,4.84,0.00,6223.95,0.0
6,MViTv2\_S\_16x4\_e,32,Train,1,99.67,0.02,574.56,0.56,1.74,0.00,3950.84,0.0
7,MViTv2\_B\_32x3,32,Train,1,99.65,0.02,809.94,0.07,1.23,0.00,5599.70,0.0
8,MViTv2\_S\_e,32,Train,2,99.87,0.01,836.33,0.14,2.39,0.00,7568.26,0.0
9,MViTv2\_S\_16x4\_e,32,Train,2,99.87,0.01,1124.43,0.07,1.78,0.00,7572.60,0.0



Inference DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv2\_S,16,Infer,1,98.00,0.00,50.69,0.05,19.73,0.02,417.83,0.0
1,MViTv2\_S\_16x4,16,Infer,1,98.00,0.00,53.94,0.04,18.54,0.01,468.83,0.0
2,MViTv2\_S,16,Infer,2,99.00,0.00,97.66,0.15,20.48,0.03,680.53,0.0
3,MViTv2\_S\_16x4,16,Infer,2,98.99,0.02,103.54,0.18,19.32,0.03,780.81,0.0
4,MViTv2\_S,16,Infer,4,99.55,0.03,191.84,0.25,20.85,0.03,1212.29,0.0
5,MViTv2\_S\_16x4,16,Infer,4,99.51,0.05,203.43,0.03,19.66,0.00,1412.77,0.0
6,MViTv2\_S,16,Infer,8,99.87,0.14,377.15,0.62,21.21,0.03,2277.22,0.0
7,MViTv2\_S\_16x4,16,Infer,8,99.95,0.01,399.79,0.17,20.01,0.01,2679.05,0.0
8,MViTv2\_S,16,Infer,16,100.00,0.00,747.74,0.07,21.40,0.00,4406.08,0.0
9,MViTv2\_S\_16x4,16,Infer,16,100.00,0.01,794.98,0.47,20.13,0.01,5208.50,0.0


### Prep for LaTeX 

In [28]:
# Drop standard deviation columns and rename for table output
mean_cols = {
    "model": "Model",
    "num_frames": "Frames",
    "batch_size": "BS",
    "gpu_util_mean": "GPU Util.",
    "latency_ms_mean": "Latency (ms)",
    "throughput_samp_per_s_mean": "Throughput (samp/s)",
    "peak_mem_mb_mean": "Peak Mem. (MB)",
}

train_mean_df = train_df[list(mean_cols.keys())].rename(columns=mean_cols)
infer_mean_df = infer_df[list(mean_cols.keys())].rename(columns=mean_cols)

def format_benchmark_df(df):
    df = df.copy()
    df["GPU Util."] = df["GPU Util."].apply(lambda x: f"{x:.2f}\\%")
    df["Latency (ms)"] = df["Latency (ms)"].apply(lambda x: f"{x:,.2f}")
    df["Throughput (samp/s)"] = df["Throughput (samp/s)"].apply(lambda x: f"{x:,.2f}")
    df["Peak Mem. (MB)"] = df["Peak Mem. (MB)"].apply(lambda x: f"{x:,.2f}")
    return df

train_mean_df = format_benchmark_df(train_mean_df)
infer_mean_df = format_benchmark_df(infer_mean_df)

display(train_mean_df)
display(infer_mean_df)

,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv2\_S,16,1,98.35\%,175.30,5.70,"1,802.01"
1,MViTv2\_S\_16x4,16,1,98.65\%,224.36,4.46,"1,800.72"
2,MViTv2\_S,16,2,99.26\%,335.94,5.95,"3,294.64"
3,MViTv2\_S\_16x4,16,2,99.47\%,430.14,4.65,"3,293.91"
4,MViTv2\_S,16,4,99.68\%,642.09,6.23,"6,224.38"
5,MViTv2\_S\_16x4,16,4,99.79\%,826.59,4.84,"6,223.95"
6,MViTv2\_S\_16x4\_e,32,1,99.67\%,574.56,1.74,"3,950.84"
7,MViTv2\_B\_32x3,32,1,99.65\%,809.94,1.23,"5,599.70"
8,MViTv2\_S\_e,32,2,99.87\%,836.33,2.39,"7,568.26"
9,MViTv2\_S\_16x4\_e,32,2,99.87\%,"1,124.43",1.78,"7,572.60"


,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv2\_S,16,1,98.00\%,50.69,19.73,417.83
1,MViTv2\_S\_16x4,16,1,98.00\%,53.94,18.54,468.83
2,MViTv2\_S,16,2,99.00\%,97.66,20.48,680.53
3,MViTv2\_S\_16x4,16,2,98.99\%,103.54,19.32,780.81
4,MViTv2\_S,16,4,99.55\%,191.84,20.85,"1,212.29"
5,MViTv2\_S\_16x4,16,4,99.51\%,203.43,19.66,"1,412.77"
6,MViTv2\_S,16,8,99.87\%,377.15,21.21,"2,277.22"
7,MViTv2\_S\_16x4,16,8,99.95\%,399.79,20.01,"2,679.05"
8,MViTv2\_S,16,16,100.00\%,747.74,21.40,"4,406.08"
9,MViTv2\_S\_16x4,16,16,100.00\%,794.98,20.13,"5,208.50"


### Print LaTeX

In [29]:
def fixhlines(txt: str) -> str:
    return (
        txt.replace("\\toprule", "\\hline")
        .replace("\\midrule", "\\hline")
        .replace("\\bottomrule", "\\hline")
    )

In [30]:
train_latex = train_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Training benchmark results for MViTv2 variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:benchmark_train",
    position="ht",
)

infer_latex = infer_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Inference benchmark results for MViTv2 variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:benchmark_infer",
    position="ht",
)

print("Training Table LaTeX:")
print(fixhlines(train_latex))
print("\nInference Table LaTeX:")
print(fixhlines(infer_latex))

Training Table LaTeX:
\begin{table}[ht]
\caption{Training benchmark results for MViTv2 variants on an NVIDIA RTX 3060 12GB GPU.}
\label{tab:benchmark_train}
\begin{tabular}{|l|c|c|r|r|r|r|}
\hline
Model & Frames & BS & GPU Util. & Latency (ms) & Throughput (samp/s) & Peak Mem. (MB) \\
\hline
MViTv2\_S & 16 & 1 & 98.35\% & 175.30 & 5.70 & 1,802.01 \\
MViTv2\_S\_16x4 & 16 & 1 & 98.65\% & 224.36 & 4.46 & 1,800.72 \\
MViTv2\_S & 16 & 2 & 99.26\% & 335.94 & 5.95 & 3,294.64 \\
MViTv2\_S\_16x4 & 16 & 2 & 99.47\% & 430.14 & 4.65 & 3,293.91 \\
MViTv2\_S & 16 & 4 & 99.68\% & 642.09 & 6.23 & 6,224.38 \\
MViTv2\_S\_16x4 & 16 & 4 & 99.79\% & 826.59 & 4.84 & 6,223.95 \\
MViTv2\_S\_16x4\_e & 32 & 1 & 99.67\% & 574.56 & 1.74 & 3,950.84 \\
MViTv2\_B\_32x3 & 32 & 1 & 99.65\% & 809.94 & 1.23 & 5,599.70 \\
MViTv2\_S\_e & 32 & 2 & 99.87\% & 836.33 & 2.39 & 7,568.26 \\
MViTv2\_S\_16x4\_e & 32 & 2 & 99.87\% & 1,124.43 & 1.78 & 7,572.60 \\
MViTv2\_B\_32x3 & 32 & 2 & 99.86\% & 1,588.39 & 1.26 & 10,733.43 \\
\h